In [1]:
import sys
import os
import mysql.connector
import pandas as pd
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')
import json
import pickle
import numpy as np
import datetime
from IPython.display import display, HTML

In [2]:
# Connect ke database config
config = get_db_config()
# Ambil host dari salah satu config (misal db_old)
print(f'Database config loaded: {config["db_old"]["host"]}')

# Connect ke DB Lama
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
print(f'Connected to old database: {config["db_old"]["database"]}')

# Connect ke DB Baru
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f'Connected to new database: {config["db_new"]["database"]}')

db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)
print(f'Connected to future database: {config["db_future"]["database"]}')


Database config loaded: localhost
Connected to old database: dataleap_v5_example_new


Connected to new database: dataleap_v5_migration
Connected to future database: dataleap_v5_migration


In [3]:
import pickle
import os

print("================================================================================")
print(" 🚀 SETUP INSERT HANDLER - FASE 1 (SISTEM MULTI-OWNER & AUTO-MAP) 🚀 ")
print("================================================================================")

# ================================================================================
# KONFIGURASI 1: LOAD MASING-MASING FILE PICKLE (TETAP TERPISAH)
# ================================================================================
data_cimut = {}
data_afrida = {}
data_hanif = {}

# 1. Load File cimut (Ganti nama file sesuai punyamu)
try:
    with open('fase_5_cimut.pkl', 'rb') as f:
        data_cimut = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik cimut.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl cimut: {e}")

# 2. Load File Afrida
try:
    with open('fase_5_afrida.pkl', 'rb') as f:
        data_afrida = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik Afrida.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl Afrida: {e}")

# 3. Load File Hanif
try:
    with open('fase_5_hanif.pkl', 'rb') as f:
        data_hanif = pickle.load(f)
    print("✓ Berhasil memuat data PKL milik Hanif.")
except Exception as e:
    print(f"⚠️ Peringatan: Gagal memuat file pkl Hanif: {e}")

print("\n================================================================================")
# ================================================================================
# KONFIGURASI 2: ISI DAFTAR TABEL MILIK MASING-MASING ORANG
# ================================================================================
list_table_cimut = [

]

list_table_afrida = [
    "presensi_siswa",
    "catatan_siswa",
    "followup_cs"
]

list_table_hanif = [
    "rapor_format",
    "rapor_format_sub",
    "rapor_format_formula",
    "rapor_format_formula_sub",
    "rapor_level_config",
    "rapor_sub_level",
    "rapor_siswa",
    "rapor_siswa_file",
    "rapor_lacak"
]

# ================================================================================
# KONFIGURASI 3: ATUR URUTAN MUTLAK PENYUNTIKAN KE DATABASE (MASTER ORDER)
# ================================================================================
# Masukkan nama tabel yang mau di-insert sesuai urutan FK (Foreign Key).
# Kamu bebas menyilangkan nama tabel di sini, sistem akan otomatis mencari pemiliknya.
master_urutan_insert = [
    "presensi_siswa",
    "catatan_siswa",
    "followup_cs",
    "rapor_format",
    "rapor_format_sub",
    "rapor_format_formula",
    "rapor_format_formula_sub",
    "rapor_level_config",
    "rapor_sub_level",
    "rapor_siswa",
    "rapor_siswa_file",
    "rapor_lacak"
]

# ================================================================================
# SISTEM DETEKTIF: MENCARI DAN MENGGABUNGKAN DATA BERDASARKAN PEMILIKNYA
# ================================================================================
print("🔍 Memulai proses pemetaan tabel ke pemilik masing-masing...\n")

data_siap_insert = {}

for table in master_urutan_insert:
    if table in list_table_cimut:
        data_siap_insert[table] = data_cimut.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data cimut.")
        
    elif table in list_table_afrida:
        data_siap_insert[table] = data_afrida.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data Afrida.")
        
    elif table in list_table_hanif:
        data_siap_insert[table] = data_hanif.get(table)
        print(f"  📦 Tabel '{table}' otomatis dipetakan dari data Hanif.")
        
    else:
        # Jika kamu memasukkan nama tabel di master_urutan tapi lupa memasukkannya di list pemilik
        data_siap_insert[table] = None
        print(f"  ❌ ERROR: Tabel '{table}' tidak ada di list cimut, Afrida, maupun Hanif!")

print("\n✅ Pemetaan selesai! Data siap disuntikkan ke database.")

 🚀 SETUP INSERT HANDLER - FASE 1 (SISTEM MULTI-OWNER & AUTO-MAP) 🚀 
⚠️ Peringatan: Gagal memuat file pkl cimut: [Errno 2] No such file or directory: 'fase_5_cimut.pkl'
✓ Berhasil memuat data PKL milik Afrida.
✓ Berhasil memuat data PKL milik Hanif.

🔍 Memulai proses pemetaan tabel ke pemilik masing-masing...

  📦 Tabel 'presensi_siswa' otomatis dipetakan dari data Afrida.
  📦 Tabel 'catatan_siswa' otomatis dipetakan dari data Afrida.
  📦 Tabel 'followup_cs' otomatis dipetakan dari data Afrida.
  📦 Tabel 'rapor_format' otomatis dipetakan dari data Hanif.
  📦 Tabel 'rapor_format_sub' otomatis dipetakan dari data Hanif.
  📦 Tabel 'rapor_format_formula' otomatis dipetakan dari data Hanif.
  📦 Tabel 'rapor_format_formula_sub' otomatis dipetakan dari data Hanif.
  📦 Tabel 'rapor_level_config' otomatis dipetakan dari data Hanif.
  📦 Tabel 'rapor_sub_level' otomatis dipetakan dari data Hanif.
  📦 Tabel 'rapor_siswa' otomatis dipetakan dari data Hanif.
  📦 Tabel 'rapor_siswa_file' otomatis dipe

## Hide code

In [4]:
import pandas as pd
import datetime
import numpy as np
import mysql.connector

# ================================================================================
# TAHAP 3: FUNGSI UTAMA INSERT (ANTI SILENT-KILLER, AUTO-BATCHING & DIAGNOSTIC)
# ================================================================================
def insert_data_with_preview_and_skip_v2(db_connection, cursor, tables_data, ordered_list, batch_size=2000):
    results = {}
    
    print("="*80)
    print("🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM AUTO-BATCHING)")
    print("="*80)
    
    # ----------------------------------------------------------------------------
    # SUB-LANGKAH A: PROSES INSERT KE MYSQL DENGAN CHUNKING (LOOPING AMAN)
    # ----------------------------------------------------------------------------
    for table_name in ordered_list:
        if table_name not in tables_data:
            results[table_name] = {'status': 'not_found', 'msg': f'⚠️  {table_name}: Tidak ditemukan di file pkl', 'warnings': []}
            continue
            
        df_target = tables_data[table_name]
        
        if df_target is None or df_target.empty:
            results[table_name] = {'status': 'empty', 'msg': f'ℹ️  {table_name}: DataFrame kosong (0 baris)', 'warnings': []}
            continue
            
        try:
                        # Bersihkan kolom kosong murni
            df_to_push = df_target.dropna(axis=1, how='all')
            
            # ================================================================
            # 🔥 KONVERSI FINAL: UBAH SEMUA KOLOM TANGGAL JADI STRING (PASTI BERHASIL!)
            # ================================================================
            for col in df_to_push.columns:
                # Deteksi apakah kolom ini bertipe datetime64 (apapun pecahannya)
                if pd.api.types.is_datetime64_any_dtype(df_to_push[col]):
                    # Ubah menjadi string format MySQL, jika kosong (NaT) jadi None
                    df_to_push[col] = df_to_push[col].apply(
                        lambda x: x.strftime('%Y-%m-%d %H:%M:%S') if pd.notnull(x) else None
                    )
            # ================================================================
            
            # Siapkan query
            columns_str = ', '.join([f'`{col}`' for col in df_to_push.columns])
            placeholders_str = ', '.join(['%s'] * len(df_to_push.columns))
            insert_query = f"INSERT IGNORE INTO `{table_name}` ({columns_str}) VALUES ({placeholders_str})"
            
            # Langsung convert ke list of tuples (tanpa ribet cleaning isna lagi, karena sudah aman)
            # Tapi kita tetap bersihkan kemungkinan ada NaN/None di kolom lain (angka/string)
            raw_data = df_to_push.to_numpy().tolist()
            clean_data_tuples = [
                tuple(None if pd.isna(x) or str(x).strip() in ["NaT", "NaN", ""] else x for x in row) 
                for row in raw_data
            ]
            
            total_rows = len(clean_data_tuples)
            actual_inserted_total = 0
            db_warnings = []
            
            # 🔥 SISTEM AUTO-BATCHING (CHUNKING) 🔥
            # Loop memotong data menjadi bagian-bagian kecil agar MySQL tidak tersedak
            for i in range(0, total_rows, batch_size):
                chunk = clean_data_tuples[i : i + batch_size]
                cursor.executemany(insert_query, chunk)
                
                # Hitung data yang berhasil masuk pada batch ini
                chunk_inserted = max(0, cursor.rowcount)
                actual_inserted_total += chunk_inserted
                
                # Jika ada yang ter-skip di batch ini, tangkap errornya (maksimal simpan 3 per tabel)
                if chunk_inserted < len(chunk) and len(db_warnings) < 3:
                    cursor.execute("SHOW WARNINGS")
                    warnings_fetched = cursor.fetchall()
                    if warnings_fetched:
                        for w in warnings_fetched:
                            w_msg = f"MySQL Warning: {w['Message']}"
                            if w_msg not in db_warnings:
                                db_warnings.append(w_msg)
                            if len(db_warnings) >= 3:
                                break
                                
                # Commit per batch agar memori stabil
                db_connection.commit()
            
            # Evaluasi Status Akhir Tabel
            if actual_inserted_total == total_rows:
                status_flag = 'success'
                msg = f'✓ {table_name}: SEMPURNA! {total_rows}/{total_rows} baris sukses masuk database.'
            else:
                status_flag = 'partial_warning'
                msg = f'⚠️ {table_name}: TER-SKIP! Dikirim {total_rows} baris, tapi yang masuk DB HANYA {actual_inserted_total} baris.'

            results[table_name] = {
                'status': status_flag, 
                'msg': msg,
                'warnings': db_warnings
            }
            
        except Exception as e:
            db_connection.rollback()
            results[table_name] = {
                'status': 'failed', 
                'msg': f'✗ {table_name}: Gagal total saat eksekusi insert - Alasan: {e}',
                'warnings': []
            }

    # ----------------------------------------------------------------------------
    # 📊 CETAK PAPAN RINGKASAN DI PALING ATAS (SUMMARY BOARD)
    # ----------------------------------------------------------------------------
    print("\n================================================================================")
    print(" 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊")
    print("================================================================================")
    
    print("🟢 TABEL YANG 100% SUKSES MASUK:")
    success_exist = False
    for table_name in ordered_list:
        res = results.get(table_name, {})
        if res.get('status') == 'success':
            print(f"  {res['msg']}")
            success_exist = True
    if not success_exist: print("  (Tidak ada tabel yang sukses sempurna)")

    print("\n🔴 TABEL YANG BERMASALAH / TER-SKIP (WAJIB DI CEK!):")
    failed_exist = False
    for table_name in ordered_list:
        res = results.get(table_name, {})
        if res.get('status') in ['failed', 'partial_warning', 'not_found', 'empty']:
            print(f"  {res['msg']}")
            
            # Cetak alasan dari MySQL (Dibatasi 3 agar tidak merusak tampilan Jupyter)
            if res.get('warnings'):
                for w_msg in res['warnings']:
                    print(f"      -> 🕵️ {w_msg}")
                    
            failed_exist = True
            
    if not failed_exist: print("  🎉 LUAR BIASA! Semua tabel bersih tidak ada data yang terbuang.")
            
    print("================================================================================\n")

    # ----------------------------------------------------------------------------
    # 📸 CETAK PREVIEW HISTORI & DIAGNOSTIK ERROR DI BAGIAN BAWAH
    # ----------------------------------------------------------------------------
    print("="*80)
    print("📸 MEMULAI LOG VISUALISASI PREVIEW & DIAGNOSTIK TABEL")
    print("="*80)
    
    for table_name in ordered_list:
        if table_name in results:
            res = results[table_name]
            
            if res['status'] == 'success':
                print(f"\n📂 [🟢 PREVIEW TABEL SUKSES: {table_name.upper()}]")
                print("-" * 50)
                display(tables_data[table_name].head(3))
                print("-" * 80)
                
            elif res['status'] in ['failed', 'partial_warning']:
                print(f"\n🚨 [🔴 DIAGNOSTIK TABEL ERROR: {table_name.upper()}] 🚨")
                print(f"Pesan Sistem: {res['msg']}")
                print("-" * 50)
                print("Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):")
                display(tables_data[table_name].head(5))
                print(f"\nTipe data Pandas untuk tabel '{table_name}':")
                print(tables_data[table_name].dtypes)
                print("-" * 80)
            
    print("\n" + "="*80)
    print("🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁")
    print("="*80)
    return results

## Output

In [5]:
# ================================================================================
# TAHAP 4: MENJALANKAN EKSEKUSI DATA REAL
# ================================================================================
results_fase_5 = insert_data_with_preview_and_skip_v2(
    db_connection=db_future, 
    cursor=cursor_future, 
    tables_data=data_siap_insert,       # <--- Menggunakan data yang sudah di-mapping otomatis
    ordered_list=master_urutan_insert   # <--- Menggunakan urutan master buatanmu
)

🎬 MEMULAI EKSEKUSI PENYUNTIKAN DATA KE MYSQL BARU (SISTEM AUTO-BATCHING)



 📊 PAPAN RINGKASAN STATUS MIGRATION DATA (SUMMARY BOARD) 📊
🟢 TABEL YANG 100% SUKSES MASUK:
  ✓ rapor_format: SEMPURNA! 41/41 baris sukses masuk database.
  ✓ rapor_format_sub: SEMPURNA! 121/121 baris sukses masuk database.
  ✓ rapor_format_formula: SEMPURNA! 3/3 baris sukses masuk database.
  ✓ rapor_format_formula_sub: SEMPURNA! 1625/1625 baris sukses masuk database.
  ✓ rapor_level_config: SEMPURNA! 340/340 baris sukses masuk database.

🔴 TABEL YANG BERMASALAH / TER-SKIP (WAJIB DI CEK!):
  ⚠️ presensi_siswa: TER-SKIP! Dikirim 108795 baris, tapi yang masuk DB HANYA 0 baris.
      -> 🕵️ MySQL Warning: Cannot add or update a child row: a foreign key constraint fails (`dataleap_v5_migration`.`presensi_siswa`, CONSTRAINT `presensi_siswa_id_jadwal_detail_foreign` FOREIGN KEY (`id_jadwal_detail`) REFERENCES `jadwal_detail` (`id_jadwal_detail`) ON DELETE SET NU)
      -> 🕵️ MySQL Warning: Cannot add or update a child row: a foreign key constraint fails (`dataleap_v5_migration`.`presensi_sis

,waktu_presensi,status_presensi,id_jadwal_detail,id_siswa
0,2023-07-05 16:24:39,1,721,347
1,2023-07-05 16:24:40,1,721,348
2,2023-07-05 16:41:58,1,331,363
3,2023-07-05 16:42:00,1,331,380
4,2023-07-05 16:42:00,1,331,381



Tipe data Pandas untuk tabel 'presensi_siswa':
waktu_presensi      datetime64[ns]
status_presensi               int8
id_jadwal_detail             int64
id_siswa                     int64
dtype: object
--------------------------------------------------------------------------------

🚨 [🔴 DIAGNOSTIK TABEL ERROR: CATATAN_SISWA] 🚨
Pesan Sistem: ⚠️ catatan_siswa: TER-SKIP! Dikirim 1535 baris, tapi yang masuk DB HANYA 0 baris.
--------------------------------------------------
Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):


,id_cs,id_jadwal,id_jadwal_detail,id_siswa,catatan_cs,id_karyawan,tanggal
0,1,9,1171,218,She's good.,None,None
1,2,9,1171,219,He's good.,None,None
2,3,9,1171,147,He's good.,None,None
3,4,20,1501,260,Jojo didn't do the task before the zoom.,None,None
4,5,20,1501,160,Vian didn't do the task before the zoom,None,None


...


Tipe data Pandas untuk tabel 'rapor_siswa':
id_jadwal             Int64
id_siswa              Int64
tanggal_input           str
id_parameter_nilai    Int64
final_result            str
dtype: object
--------------------------------------------------------------------------------

🚨 [🔴 DIAGNOSTIK TABEL ERROR: RAPOR_SISWA_FILE] 🚨
Pesan Sistem: ⚠️ rapor_siswa_file: TER-SKIP! Dikirim 1499 baris, tapi yang masuk DB HANYA 0 baris.
--------------------------------------------------
Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):


,id_rapor_siswa,file_rapor_path
0,5166,uploads/rapor/S0000329.jpeg
1,5172,uploads/rapor/S0000474.jpeg
2,5183,uploads/rapor/S0000481.jpeg
3,5178,uploads/rapor/S0000475.jpeg
4,5189,uploads/rapor/S0000482.jpeg



Tipe data Pandas untuk tabel 'rapor_siswa_file':
id_rapor_siswa     Int64
file_rapor_path      str
dtype: object
--------------------------------------------------------------------------------

🚨 [🔴 DIAGNOSTIK TABEL ERROR: RAPOR_LACAK] 🚨
Pesan Sistem: ⚠️ rapor_lacak: TER-SKIP! Dikirim 1366 baris, tapi yang masuk DB HANYA 0 baris.
--------------------------------------------------
Berikut cuplikan data yang kemungkinan ditolak MySQL (Cek FK dan Tipe Data):


,id_siswa,id_jadwal,tanggal_terkirim,status_pengiriman,id_rapor_siswa_file
0,507,133,2024-11-18 10:57:59,Terkirim,395
1,137,312,2024-12-02 11:37:11,Terkirim,398
2,596,312,2024-12-02 11:37:12,Terkirim,399
3,335,312,2024-12-02 11:37:12,Terkirim,396
4,466,312,2024-12-02 11:37:13,Terkirim,397



Tipe data Pandas untuk tabel 'rapor_lacak':
id_siswa               Int64
id_jadwal              Int64
tanggal_terkirim         str
status_pengiriman        str
id_rapor_siswa_file    Int64
dtype: object
--------------------------------------------------------------------------------

🏁 PROSES INSPEKSI SELESAI. SILAKAN CEK HASIL DIAGNOSTIK DI ATAS 🏁


In [6]:
# print("================================================================================")
# print(" 🧹 MEMULAI PROSES TRUNCATE DATA GLOBAL - FASE 1 (SISTEM RINGKASAN ATAS) 🧹 ")
# print("================================================================================")

# def truncate_tables_with_summary(db_connection, cursor, ordered_list):
#     truncate_results = {}
    
#     try:
#         # 🔥 SAKTI 1: Matikan benteng Foreign Key checks agar MySQL tidak memblokir penghapusan
#         cursor.execute("SET FOREIGN_KEY_CHECKS=0")
#         db_connection.commit()
#         print("🔓 Sensor Foreign Key Checks berhasil DIMATIKAN sementara.\n")
#     except Exception as e:
#         print(f"✗ Gagal mematikan Foreign Key Checks: {e}")
#         return
        
#     # ----------------------------------------------------------------------------
#     # SUB-LANGKAH A: PROSES EKSEKUSI TRUNCATE DI BELAKANG LAYAR
#     # ----------------------------------------------------------------------------
#     for table_name in ordered_list:
#         try:
#             truncate_query = f"TRUNCATE TABLE `{table_name}`"
#             cursor.execute(truncate_query)
#             db_connection.commit()
            
#             truncate_results[table_name] = {
#                 'status': 'success',
#                 'msg': f"✓ {table_name}: Sukses dibersihkan total! Seluruh baris data amblas."
#             }
#         except Exception as e:
#             db_connection.rollback()
#             truncate_results[table_name] = {
#                 'status': 'failed',
#                 'msg': f"✗ {table_name}: Gagal dikosongkan! Alasan: {e}"
#             }

#     try:
#         # 🔥 SAKTI 2: Wajib nyalakan kembali benteng Foreign Key checks setelah selesai
#         cursor.execute("SET FOREIGN_KEY_CHECKS=1")
#         db_connection.commit()
#         print("🔒 Sensor Foreign Key Checks berhasil DIHIDUPKAN kembali dengan aman.")
#     except Exception as e:
#         print(f"⚠️ Peringatan: Gagal menghidupkan kembali Foreign Key Checks: {e}")

#     # ----------------------------------------------------------------------------
#     # 🔥 CETAK PAPAN RINGKASAN TRUNCATE DI PALING ATAS (ANTI-SCROLL BOARD)
#     # ----------------------------------------------------------------------------
#     print("\n================================================================================")
#     print(" 📊 PAPAN RINGKASAN STATUS TRUNCATE DATABASE (CLEANUP SUMMARY BOARD) 📊")
#     print("================================================================================")
#     for table_name in ordered_list:
#         if table_name in truncate_results:
#             print(truncate_results[table_name]['msg'])
#         else:
#             print(f"⚠️  {table_name}: Lewat dari antrean pembersihan.")
#     print("================================================================================")
    
#     return truncate_results

# # === JALANKAN EKSEKUSI PEMBERSIHAN MENGGUNAKAN DAFTAR TABEL SALING SILANGMU ===
# results_truncate_fase_5 = truncate_tables_with_summary(
#     db_connection=db_new, 
#     cursor=cursor_new, 
#     ordered_list=tables_to_insert_ordered  # Otomatis memakai list urutan saling silang yang kita buat tadi
# )